# Bridging Neural Voice and Biological Audition
### Ears, Audition, and Language Models

This notebook explores how the deep learning architectures found in modern **Neural Voice** models (like `tiny-aya`, `Moshi`, and `Pocket TTS`) map directly onto biological cognitive structures.

Specifically, we will visualize:
1. **The Ear (Frontend):** Time-frequency feature extraction in the cochlea.
2. **Cochlear Self-Attention:** Top-down efferent feedback tuning sensitivity to specific auditory "objects" (e.g., speech formants), similar to DINO's spatial object attention in vision.
3. **The Auditory Nerve (Mimi Codec):** Discretizing continuous sensory streams into codebooks (neural spike trains).
4. **The Language Cortex (Cohere Backbone):** Parallel processing streams for listening (user audio) and planning/speaking (model audio).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# Apply stunning dark theme
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.facecolor': '#0d1117',
    'figure.facecolor': '#0d1117',
    'axes.edgecolor': '#30363d',
    'grid.color': '#21262d',
    'text.color': '#c9d1d9',
    'xtick.color': '#c9d1d9',
    'ytick.color': '#c9d1d9',
    'axes.labelcolor': '#c9d1d9',
    'font.family': 'sans-serif'
})

# Custom colormaps for attention and biology
cmap_cochlea = sns.color_palette("magma", as_cmap=True)
cmap_attention = sns.color_palette("mako", as_cmap=True)
cmap_spikes = LinearSegmentedColormap.from_list('spikes', ['#0d1117', '#58a6ff'])

print("Environment setup complete. Ready to explore Neural Voice.")

## 1. The Ear & Cochlea (Frontend)

In human biology, the basilar membrane in the inner ear acts as a physical Fourier transformer, breaking complex acoustic waves into distinct frequency bands over time. 

In our **Audition Lab**, this is equivalent to passing raw audio through a **Learnable Audio Transformer** frontend (or a Mel-filterbank). Let's simulate a complex acoustic signal (e.g., a spoken word with distinct formants) and visualize its Cochleagram.

In [ ]:
np.random.seed(42)
time_steps = 200
freq_bins = 64

# Simulate a cochleagram: Background noise + 3 speech formants
cochleagram = np.random.rand(freq_bins, time_steps) * 0.2

# Formant 1 (Low freq, long duration)
cochleagram[10:15, 20:180] += np.hanning(160) * 0.8
# Formant 2 (Mid freq, changing pitch)
formant_2_path = np.linspace(30, 40, 100).astype(int)
for i, f in enumerate(formant_2_path):
    cochleagram[f-2:f+3, 40+i] += 0.9
# Formant 3 (High freq, transient burst - e.g., a consonant)
cochleagram[50:60, 40:55] += np.hanning(15) * 1.0

# Add some smoothing
from scipy.ndimage import gaussian_filter
cochleagram = gaussian_filter(cochleagram, sigma=0.8)

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(cochleagram, aspect='auto', origin='lower', cmap=cmap_cochlea)
ax.set_title("Simulated Cochleagram (Time-Frequency Representation)", fontsize=14, weight='bold', color='#58a6ff')
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Frequency Channel (Basilar Membrane Position)")
fig.colorbar(im, ax=ax, label="Acoustic Energy")
plt.tight_layout()
plt.show()

## 2. Cochlear Self-Attention (Top-Down Tuning)

In the `everyday-sketch/engineering/perception/vision` repository, you explored how Vision Transformers (like DINO) use self-attention to segment objects. 

In audition, the brain uses **efferent feedback pathways** to physically tune the hair cells in the cochlea, focusing attention on specific sounds (the "cocktail party effect"). An Audio Transformer applies self-attention across the time-frequency patches.

Let's visualize how different attention heads might bind to the distinct acoustic objects (formants and transients) we created above.

In [ ]:
# Simulate 3 Attention Heads binding to different features
head_1 = np.zeros_like(cochleagram) # Attends to the transient/consonant
head_1[45:65, 30:65] = cochleagram[45:65, 30:65] * 2.5

head_2 = np.zeros_like(cochleagram) # Attends to the gliding formant
for i, f in enumerate(formant_2_path):
    head_2[f-5:f+5, 30+i:50+i] = cochleagram[f-5:f+5, 30+i:50+i] * 1.5

head_3 = np.zeros_like(cochleagram) # Attends to background / sustained low frequency
head_3[5:20, 10:190] = cochleagram[5:20, 10:190] * 1.2

heads = [gaussian_filter(h, sigma=1.2) for h in [head_1, head_2, head_3]]
titles = ["Head 1: Transient Focus (Consonants)", "Head 2: Pitch Tracking (Vowels)", "Head 3: Sustained Energy"]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for i, ax in enumerate(axes):
    # Overlay attention on original cochleagram as a contour or heatmap
    ax.imshow(cochleagram, aspect='auto', origin='lower', cmap='gray', alpha=0.3)
    im = ax.imshow(heads[i], aspect='auto', origin='lower', cmap=cmap_attention, alpha=0.8)
    ax.set_title(titles[i], color='#79c0ff')
    ax.axis('off')

plt.suptitle("Cochlear Self-Attention Maps (Audio Transformer Heads)", fontsize=16, y=1.05, weight='bold', color='#58a6ff')
plt.tight_layout()
plt.show()

## 3. The Auditory Nerve (Mimi Codec & Discretization)

The `tiny-aya` and Moshi models rely on the **Mimi Codec** to quantize continuous audio embeddings into discrete tokens (Codebooks CB0-CB7) operating at a specific frame rate (e.g. 12.5 Hz).

Biologically, this is the **Auditory Nerve**, which translates continuous acoustic pressure into discrete, all-or-nothing action potentials (spike trains) that travel to the cortex. Let's simulate the quantization of our attended acoustic features into a discrete codebook sequence.

In [ ]:
# Collapse the time-frequency attention maps into discrete time steps (downsampling)
# Simulate 20 discrete time frames
downsampled_time = np.linspace(0, time_steps-1, 20, dtype=int)
codebook_spikes = np.zeros((8, 20)) # 8 Codebooks (CB0 to CB7)

# Map continuous energy to discrete tokens (mocking quantization)
for i, t in enumerate(downsampled_time):
    # CB0 tracks Head 1 (transients)
    codebook_spikes[0, i] = np.mean(head_1[:, t-5:t+5]) > 0.1
    # CB1 tracks Head 2 (formants)
    codebook_spikes[1, i] = np.mean(head_2[:, t-5:t+5]) > 0.1
    # CB2 tracks Head 3 (low freq)
    codebook_spikes[2, i] = np.mean(head_3[:, t-5:t+5]) > 0.1
    # Random neural noise for other codebooks
    codebook_spikes[3:, i] = np.random.rand(5) > 0.85

fig, ax = plt.subplots(figsize=(12, 3))
# Plot spike trains
sns.heatmap(codebook_spikes, cmap=cmap_spikes, cbar=False, linewidths=1, linecolor='#0d1117', ax=ax)
ax.set_title("Discrete Codebook Representation (Auditory Nerve Spike Trains)", fontsize=14, weight='bold', color='#58a6ff')
ax.set_xlabel("Time Step (Token Frame)")
ax.set_ylabel("Codebook (CB0 - CB7)")
ax.set_yticks(np.arange(8) + 0.5)
ax.set_yticklabels([f"CB{i}" for i in range(8)], rotation=0)
plt.tight_layout()
plt.show()

## 4. The Language Cortex (Parallel Streams)

In Wernicke's and Broca's areas (language reception and production), the brain is constantly listening to the environment while simultaneously planning internal speech.

The **Cohere2 3B Backbone** in `tiny-aya` handles this beautifully through a **Parallel Two-Stream Format**:
- `User Audio Stream`: The incoming tokens we just processed from the ear (Listening).
- `Model Audio Stream`: The outgoing tokens the model is planning to articulate (Speaking).

The model learns turn-taking and semantic meaning by processing these side-by-side.

In [ ]:
# Simulate parallel streams over 40 time steps
total_frames = 40
user_stream = np.zeros(total_frames)
model_stream = np.zeros(total_frames)

# User speaks from frame 5 to 15
user_stream[5:15] = np.random.uniform(0.5, 1.0, 10)
# Model plans response (internal state increases) during user speech
internal_planning = np.zeros(total_frames)
internal_planning[10:20] = np.linspace(0.1, 0.9, 10)
# Model speaks from frame 20 to 35
model_stream[20:35] = np.random.uniform(0.5, 1.0, 15)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(user_stream, label="User Audio Stream (Listening)", color='#79c0ff', lw=3, drawstyle='steps-mid')
ax.plot(model_stream, label="Model Audio Stream (Speaking)", color='#ff7b72', lw=3, drawstyle='steps-mid')
ax.plot(internal_planning, label="Internal State (Semantic Planning)", color='#d2a8ff', lw=2, linestyle='--')

# Highlight turn-taking
ax.axvspan(5, 15, color='#79c0ff', alpha=0.1)
ax.axvspan(20, 35, color='#ff7b72', alpha=0.1)

ax.set_title("Parallel Language Streams (Cohere2 / Cortex Dynamics)", fontsize=14, weight='bold', color='#58a6ff')
ax.set_xlabel("Time Frame")
ax.set_ylabel("Activity Level")
ax.legend(facecolor='#0d1117', edgecolor='#30363d')
ax.grid(True, axis='x', color='#21262d', linestyle='--')
plt.tight_layout()
plt.show()